In [10]:
import pandas as pd

df_raw = pd.read_csv('/content/enhanced_health_insurance_claims.csv')

print(df_raw.shape)
print(df_raw.columns.tolist())
display(df_raw.head())
df = df_raw.copy()

(4500, 17)
['ClaimID', 'PatientID', 'ProviderID', 'ClaimAmount', 'ClaimDate', 'DiagnosisCode', 'ProcedureCode', 'PatientAge', 'PatientGender', 'ProviderSpecialty', 'ClaimStatus', 'PatientIncome', 'PatientMaritalStatus', 'PatientEmploymentStatus', 'ProviderLocation', 'ClaimType', 'ClaimSubmissionMethod']


,ClaimID,PatientID,ProviderID,ClaimAmount,ClaimDate,DiagnosisCode,ProcedureCode,PatientAge,PatientGender,ProviderSpecialty,ClaimStatus,PatientIncome,PatientMaritalStatus,PatientEmploymentStatus,ProviderLocation,ClaimType,ClaimSubmissionMethod
0,10944daf-f7d5-4e1d-8216-72ffa609fe41,8552381d-7960-4f64-b190-b20b8ada00a1,4a4cb19c-4863-41cf-84b0-c2b21aace988,3807.95,2024-06-07,yy006,hd662,16,M,Cardiology,Pending,90279.43,Married,Retired,Jameshaven,Routine,Paper
1,fcbebb25-fc24-4c0f-a966-749edcf83fb1,327f43ad-e3bd-4473-a9ed-46483a0a156f,422e02dd-c1fd-43dd-8af4-0c3523f997b1,9512.07,2023-05-30,tD052,mH831,27,M,Pediatrics,Approved,130448.02,Single,Student,Beltrantown,Routine,Online
2,9e9983e7-9ea7-45f5-84d8-ce49ccd8a4a1,6f3acdf7-73aa-4afa-9c2e-b25b27bdb5b0,f7733b3f-0980-47b5-a7a0-ee390869355b,7346.74,2022-09-27,zx832,dg637,40,F,Cardiology,Pending,82417.54,Divorced,Employed,West Charlesport,Emergency,Online
3,a06273ed-44bb-452b-bbad-8618de080494,5d58e183-701e-406c-a8c6-5b73cac5e912,f7a04581-de96-44ee-b773-8adac02baa59,6026.72,2023-06-25,kr421,kG326,65,M,Neurology,Pending,68516.96,Widowed,Student,West Aprilhaven,Routine,Phone
4,f702a717-254b-4cff-a0c7-8395db2f6616,8a8ebdf6-3af0-4f14-82f3-37b937c3d270,b80b9e77-97f0-47d7-b561-19f9658a7bdf,1644.58,2023-07-24,LZ261,cx805,24,M,General Practice,Pending,84122.17,Married,Student,Lake Michele,Inpatient,Phone


In [11]:
import numpy as np
np.random.seed(42)  #will generate same results on re-run

In [12]:
# Cleaning rows
df['ClaimDate'] = pd.to_datetime(df['ClaimDate'])
df['ClaimAmount'] = pd.to_numeric(df['ClaimAmount'], errors='coerce')
df = df.dropna(subset=['ClaimAmount', 'ClaimDate'])


In [13]:
df = df[df['PatientEmploymentStatus'] == 'Employed'].copy()
print("Employed claims:", df.shape)

Employed claims: (1188, 17)


In [14]:
#Adding EmployerGroup column with random employer values
employers = [f"Employer_{i:02d}" for i in range(1, 19)]
df['EmployerGroup'] = np.random.choice(employers, size=len(df))

#Adding plan tier column
df['PlanTier'] = np.random.choice(['Bronze', 'Silver', 'Gold'], size=len(df), p=[0.3, 0.45, 0.25])

In [15]:
#Depend status function and column
def assign_dependents(marital):
    if marital == 'Married':
        return np.random.choice(['Employee+Spouse', 'Family'], p=[0.5, 0.5])
    elif marital in ['Divorced', 'Widowed']:
        return np.random.choice(['Employee Only', 'Family'], p=[0.7, 0.3])
    else:  # Single
        return np.random.choice(['Employee Only'], p=[1.0])

df['DependentStatus'] = df['PatientMaritalStatus'].apply(assign_dependents)

In [16]:
#policy year from claim date
df['PolicyYear'] = df['ClaimDate'].dt.year

print(df[['EmployerGroup','PlanTier','DependentStatus','PolicyYear']].head())
print(df['PolicyYear'].value_counts())

   EmployerGroup PlanTier  DependentStatus  PolicyYear
2    Employer_07   Bronze    Employee Only        2022
11   Employer_15   Silver           Family        2024
16   Employer_11   Bronze  Employee+Spouse        2023
20   Employer_08     Gold    Employee Only        2023
24   Employer_07     Gold    Employee Only        2023
PolicyYear
2023    599
2024    303
2022    286
Name: count, dtype: int64


In [17]:
#cleaned & modified dataset
df.to_csv('clean_enriched_claims.csv', index=False)

from google.colab import files
files.download('clean_enriched_claims.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>